## Drone Localisation Simulation
This project is setup to test the simulation of a microphone picking up the sounds of a drone for the purposes of localisation.



In [ ]:
# !pip install pyroomacoustics numpy scipy==1.11.1 matplotlib acoular

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyroomacoustics as pra
from numpy import hamming
from scipy.io import wavfile
from IPython.display import Audio, display
from math import floor
import acoular
import os
import h5py
import tables
import time

## Building the Space

This is where we specify the dimensions of our simulation environment.

In [ ]:
import numpy as np

corner_locs = 150.0
metric_conv = 0.3048

corners = np.array(
    [
        [0.0,0.0],
        [corner_locs,0.0],
        [corner_locs,corner_locs],
        [0.0,corner_locs]
    ]
).T
h = 9.5

# h *= 0.3048
# corners *= 0.3048

source = np.array([[122.45], [122], [4.0]])
# source = np.array([[50], [0], [4.0]])
source_noise = np.array([[114], [40], [2.0]])


grid_side_length = 5  # Number of microphones per side i.e 4 = 16 total

# Define the spacing between microphones in the grid (e.g., 1 unit)
mic_spacing = 0.15

# Define the starting point of your grid (e.g., offset from origin)
start_x = 1.0
start_y = 1.0

# Fixed Z-coordinate for all microphones
fixed_z = 1.2

# Generate X and Y coordinates for the grid using np.linspace
# np.linspace(start, stop, num_points) creates evenly spaced numbers over an interval.
x_coords_1d = np.linspace(start_x, start_x + (grid_side_length - 1) * mic_spacing, grid_side_length)
y_coords_1d = np.linspace(start_y, start_y + (grid_side_length - 1) * mic_spacing, grid_side_length)

# Use np.meshgrid to create the 2D grid of (X, Y) pairs
X_grid, Y_grid = np.meshgrid(x_coords_1d, y_coords_1d)

# Flatten the grids into 1D arrays of all X and all Y coordinates
all_x_coords = X_grid.flatten()
all_y_coords = Y_grid.flatten()

# Create the Z coordinates (all fixed at 'fixed_z')
all_z_coords = np.full(all_x_coords.shape[0], fixed_z)

# Preferred and most readable way:
mic_locs = np.array([all_x_coords, all_y_coords, all_z_coords])

# corner_locs *= metric_conv
# source *= metric_conv
# source_noise *= metric_conv
# mic_locs *= metric_conv

fs, s = wavfile.read("drone_audio.wav")

## Displaying the Space

In [ ]:
# Display the Space in 2D
room = pra.Room.from_corners(corners)
room.add_source(source[:2])
room.add_source(source_noise[:2])
room.add_microphone_array(pra.MicrophoneArray(mic_locs[:2,:], fs=fs))

fig, ax = room.plot(img_order=2)
ax.set_xlim([-1, 1 + corner_locs])
ax.set_ylim([-1, 1 + corner_locs])


In [ ]:
# Set wall materials for open field
wall_material = pra.Material(energy_absorption=1.0, scattering=0.0)
ceiling_material = pra.Material(energy_absorption=1.0, scattering=0.0)
floor_material = pra.Material(energy_absorption=1.0, scattering=0.0)

# Redefine room for 3D raytracing simulation.
room = pra.Room.from_corners(corners, fs=fs, max_order=5, materials=wall_material, ray_tracing=True, air_absorption=True)
room.extrude(h, materials=ceiling_material)
room.set_ray_tracing(receiver_radius=0.1, n_rays=10000, energy_thres=1e-7)
room.add_source(source)
room.add_microphone_array(pra.MicrophoneArray(mic_locs, fs=fs))

# Compute image sources
room.image_source_model()
room.plot_rir()
fig = plt.gcf()
fig.set_size_inches(20, 10)

t60 = pra.experimental.measure_rt60(room.rir[0][0], fs=room.fs, plot=False)
print(f"The RT60 is {t60 * 1000:.0f} ms")

In [ ]:
# The desired reverberation time and dimensions of the room
rt60_tgt = 1 # seconds
room_dim = [corner_locs,corner_locs,h] # meters

# import a mono wavfile as the source signal
# the sampling frequency should match that of the room
fs, audio = wavfile.read("drone_audio.wav")
fs, audio_noise = wavfile.read("noise.wav")

# noise
noise = True
noise_attenuation = 0.05
audio_noise = audio_noise * noise_attenuation

if audio.ndim > 1:
    audio = np.mean(audio, axis=1)

if audio_noise.ndim > 1:
    audio_noise = np.mean(audio_noise, axis=1)

# We invert Sabine's formula to obtain the parameters for the ISM simulator
e_absorption, max_order = pra.inverse_sabine(rt60_tgt, room_dim)


# Create the room
room = pra.ShoeBox(
    room_dim, fs=fs, materials=pra.Material(1.0), max_order=max_order
)


# place the source in the room
room.add_source(source, signal=audio, delay=0.5)
if noise:
  room.add_source(source_noise, signal=audio_noise, delay=0.25)


# finally place the array in the room
room.add_microphone_array(mic_locs)

# Run the simulation (this will also build the RIR automatically)
room.simulate()

"""
room.mic_array.to_wav(
    f"/content/mic_output.wav",
    norm=True,
    bitdepth=np.int16,
)
"""

# measure the reverberation time
rt60 = room.measure_rt60()
print("The desired RT60 was {}".format(rt60_tgt))
print("The measured RT60 is {}".format(rt60[1, 0]))

# Create a plot
plt.figure()

# plot one of the RIR. both can also be plotted using room.plot_rir()
rir_1_0 = room.rir[1][0]

plt.plot(np.arange(len(rir_1_0)) / room.fs, rir_1_0)
plt.title("The RIR from source 0 to mic 1")
plt.xlabel("Time [s]")
plt.show()

# plot signal at microphone 1

plt.plot(audio)
plt.title("Raw signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(audio_noise)
plt.title("Noise")
plt.xlabel("Time [s]")
plt.show()


plt.plot(room.mic_array.signals[0, :])
plt.title("Microphone 1 Signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(room.mic_array.signals[1, :])
plt.title("Microphone 2 Signal")
plt.xlabel("Time [s]")
plt.show()

plt.plot(room.mic_array.signals[2, :])
plt.title("Microphone 3 Signal")
plt.xlabel("Time [s]")
plt.show()



audio_data1 = room.mic_array.signals[0, :]
audio_data2 = room.mic_array.signals[1, :]
print("Unaltered Drone Sound: Normalized")
display(Audio(data=audio, rate=fs))
print("Unaltered Noise Sound: Normalized")
display(Audio(data=audio_noise, rate=fs))
print("Microphone 1: Normalized")
display(Audio(data=audio_data1, rate=fs))
print("Microphone 2: Normalized")
display(Audio(data=audio_data2, rate=fs))

In [ ]:
acoular.demo.acoular_demo.run()

Apt install follow gazebo and ROS2

## Method to delete h5 file

Due to the unique properties h5 files you will have difficulties editing them without special considerations. The easier method is to recreate this h5 file every time. If you have not run this code you do not need to run the next code block

In [ ]:
files = ["sim_signals.h5", "three_sources.h5"]

for file_name in files:
    if os.path.exists(file_name):
        print(f"File '{file_name}' exists. Attempting to close any open HDF5/PyTables handles...")

        try:
            # This is the most effective way to close HDF5 files opened by this process
            # (via h5py or PyTables).
            tables.file._open_files.close_all()
            print("Successfully attempted to close all open HDF5/PyTables files within this process.")
        except ImportError:
            print("PyTables is not installed, skipping tables.file._open_files.close_all().")
            print("If you frequently encounter file locking issues, consider 'pip install tables'.")
        except Exception as e:
            print(f"Error while trying to close HDF5/PyTables files: {e}")

        # Give a brief moment for the OS to release the handle
        time.sleep(0.1)

        # Now, attempt to delete the file
        try:
            os.remove(file_name)
            print(f"File '{file_name}' deleted successfully.")
        except OSError as e:
            print(f"Error deleting file '{file_name}': {e}")
            print("This often indicates another *external* process (not this Python script) is holding the file open.")
            print("Please ensure no other applications (like HDFView, another Python script, etc.) are using 'sim_signals.h5'.")
    else:
        print(f"File '{file_name}' does not exist.")

In [ ]:
import scipy.fft as fft

N = len(audio)
T = 1.0 / fs
yf = fft.fft(audio)
xf = fft.fftfreq(N, T)[:N//2]

plt.figure(figsize=(10, 4))
plt.plot(xf, 2.0/N * np.abs(yf[0:N//2]))
plt.title("FFT of Drone Audio Signal")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.xlim(0, 16000) # Focus on relevant range
plt.grid(True)
plt.show()

## Localisation using Acoular

In [ ]:

# Use mic positions directly (shape must be (3, N))
mic_positions = room.mic_array.R  # shape: (3, num_mics)

mg = acoular.MicGeom()
mg.pos_total = mic_positions  # no need for .T, Acoular expects (3, N)

print("MicGeom initialized with", mg.num_mics, "mics")


In [ ]:
import h5py
import os

mic_signals = room.mic_array.signals  # Or however you've stored it
sample_freq = fs  # or whatever you've used

with h5py.File('sim_signals.h5', 'w') as f:
    f.create_dataset('time_data', data=mic_signals.T)  # Transpose to shape (samples, channels)
    f.attrs['sample_freq'] = sample_freq


In [ ]:
with h5py.File('sim_signals.h5', 'r') as f:
    data = f['time_data']
    print("Shape of time_data:", data.shape)

In [ ]:
import h5py

# Set your actual sampling frequency
SAMPLE_FREQ = fs  # or whatever value was used in Pyroomacoustics

with h5py.File("sim_signals.h5", "r+") as f:
    if 'time_data' in f:
        f['time_data'].attrs['sample_freq'] = SAMPLE_FREQ
        print("Added 'sample_freq' =", SAMPLE_FREQ)
    else:
        print("'/time_data' dataset not found.")

In [ ]:
from pathlib import Path
import acoular as ac
import matplotlib.pyplot as plt

search_freq = 5000
plot_size = int(corner_locs)

# Load mic geometry and signals
ts = ac.TimeSamples(file='sim_signals.h5')
ps = ac.PowerSpectra(source=ts, block_size=512, window='Hanning')

# Define scanning grid around expected source location
rg = ac.RectGrid(x_min=-1, x_max=plot_size, y_min=-1, y_max=plot_size, z=2, increment=0.05)
st = ac.SteeringVector(grid=rg, mics=mg)
bb = ac.BeamformerBase(freq_data=ps, steer=st)

# Beamform at a selected frequency (e.g., 8000 Hz)
pm = bb.synthetic(search_freq, 1) # Frequency of interest, octave range
Lm = ac.L_p(pm)

plt.imshow(Lm.T, origin='lower', vmin=Lm.max()-3, extent=rg.extend(), interpolation='bicubic')
plt.plot(source[0,0], source[1,0], 'x', color='red', markersize=10, label='Actual Source')
plt.title("Beamforming Map")
plt.colorbar(label='dB')
plt.show()


In [ ]:
plt.imshow(Lm.T, origin='lower', vmin=Lm.max()-0.2, extent=rg.extend(), interpolation='bicubic')
plt.plot(source[0,0], source[1,0], 'x', color='red', markersize=10, label='Actual Source')
plt.title("Beamforming Map")
plt.colorbar(label='dB')
plt.show()